# Lesson 02: The Reverse Denoising Process

We derive and implement the reverse posterior $q(x_{t-1} | x_t, x_0)$ that enables denoising. If we can predict $x_0$ from $x_t$, we can compute the reverse step.

**Paper reference:** D3PM (Austin et al., 2021) Section 3.2.

In [ ]:
%pip install torch numpy matplotlib --quiet

In [ ]:
import sys
sys.path.insert(0, '../../..')

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from shared.utils.seed import set_seed
from shared.datasets.text import SimpleTokenizer

# Import our forward process from Lesson 01
sys.path.insert(0, '../lesson01-forward-corruption-process')
from src.forward_process import DiscreteForwardProcess

set_seed(42)

## 1. The Reverse Posterior: Intuition

Given corrupted tokens $x_t$ and knowledge of the original $x_0$, we can compute the posterior:

$$q(x_{t-1} = j \mid x_t, x_0) \propto \underbrace{Q_t[j, x_t]}_{\text{likelihood}} \cdot \underbrace{\bar{Q}_{t-1}[x_0, j]}_{\text{prior}}$$

- **Likelihood:** How likely is the observed $x_t$ if $x_{t-1}$ were token $j$?
- **Prior:** How likely is $x_{t-1} = j$ given we started from $x_0$?

In [ ]:
from src.reverse_process import compute_posterior, sample_reverse_step

# Setup: small vocabulary for visualization
vocab_size = 6  # tokens: 0=<pad>, 1=<unk>, 2=<mask>, 3=A, 4=B, 5=C
T = 50
fp = DiscreteForwardProcess(vocab_size, T, schedule="absorbing", mask_token_id=2)

# Clean sequence: [A, B, C] = [3, 4, 5]
x_0 = torch.tensor([[3, 4, 5]])

# Corrupt to t=25
t_val = 25
t = torch.tensor([t_val])
x_t = fp.sample_q_t(x_0, t)
print(f"x_0: {x_0[0].tolist()}")
print(f"x_t (t={t_val}): {x_t[0].tolist()}")

In [ ]:
# Compute posterior using ORACLE (we know x_0)
x_0_probs = F.one_hot(x_0.long(), vocab_size).float()  # perfect knowledge
posterior = compute_posterior(x_t, x_0_probs, t_val, fp)

print("Posterior q(x_{t-1} | x_t, x_0):")
for pos in range(3):
    print(f"  Position {pos}: {posterior[0, pos].numpy().round(4)}")
    print(f"    x_t={x_t[0, pos].item()}, x_0={x_0[0, pos].item()}, argmax={posterior[0, pos].argmax().item()}")

## 2. Oracle Reverse Process

Let's verify that with perfect knowledge of $x_0$, the reverse process recovers the original sequence.

In [ ]:
from src.reverse_process import demo_reverse_with_oracle

# Longer sequence for a better test
texts = ["the cat sat on the mat"]
tokenizer = SimpleTokenizer(texts, level="char")

x_0 = torch.tensor([tokenizer.encode(texts[0])])
fp_text = DiscreteForwardProcess(
    vocab_size=tokenizer.vocab_size,
    num_timesteps=50,
    schedule="absorbing",
    mask_token_id=tokenizer.mask_id,
)

recovered = demo_reverse_with_oracle(x_0, fp_text, t_start=20)

print(f"\nOriginal: '{tokenizer.decode(x_0[0].tolist())}'")
print(f"Recovered: '{tokenizer.decode(recovered[0].tolist())}'")

match_rate = (recovered[0] == x_0[0]).float().mean().item()
print(f"Match rate: {match_rate:.1%}")

## 3. What Happens Without an Oracle?

In practice, we don't know $x_0$ — we need a neural network to predict it. Let's see what happens with a **noisy** prediction.

In [ ]:
# Simulate a "bad" model: mix true x_0 with uniform noise
x_0_onehot = F.one_hot(x_0.long(), tokenizer.vocab_size).float()

noise_levels = [0.0, 0.3, 0.6, 0.9]
for noise in noise_levels:
    # Mix oracle with uniform noise
    uniform = torch.ones_like(x_0_onehot) / tokenizer.vocab_size
    noisy_pred = (1 - noise) * x_0_onehot + noise * uniform
    
    # Corrupt then try to reverse
    t_start = 20
    t = torch.tensor([t_start])
    x_t = fp_text.sample_q_t(x_0, t)
    
    x_current = x_t
    for t_val in range(t_start, 0, -1):
        x_current = sample_reverse_step(x_current, noisy_pred, t_val, fp_text)
    
    match = (x_current[0] == x_0[0]).float().mean().item()
    decoded = tokenizer.decode(x_current[0].tolist())
    print(f"Noise={noise:.1f}: match={match:.1%}, result='{decoded}'")

## 4. Temperature Sampling

We can control the diversity of samples by adjusting the temperature of the $x_0$ prediction.

In [ ]:
from src.reverse_process import sample_reverse_step_with_temperature

# Use the oracle for a clean comparison
t_start = 30
temperatures = [0.1, 0.5, 1.0, 2.0]

for temp in temperatures:
    results = []
    for trial in range(5):
        t = torch.tensor([t_start])
        x_t = fp_text.sample_q_t(x_0, t)
        x_current = x_t
        x_0_probs = F.one_hot(x_0.long(), tokenizer.vocab_size).float()
        for t_val in range(t_start, 0, -1):
            x_current = sample_reverse_step_with_temperature(
                x_current, x_0_probs, t_val, fp_text, temperature=temp
            )
        results.append(tokenizer.decode(x_current[0].tolist()))
    
    print(f"Temperature={temp}:")
    for r in results[:3]:
        print(f"  '{r}'")
    print()

## Key Takeaways

1. The reverse posterior $q(x_{t-1} | x_t, x_0)$ combines a **likelihood** term (from $Q_t$) and a **prior** term (from $\bar{Q}_{t-1}$).
2. If we can predict $x_0$ from $x_t$, we can compute this posterior exactly.
3. Better $x_0$ predictions lead to better reverse samples — this is why training the denoiser matters.
4. Temperature controls the diversity-quality tradeoff.

## What's Next

In Lesson 03, we build a **complete D3PM** model with a transformer denoiser, loss function, and sampling loop.